# Features for Churn Prediction

**Goal:** for every customer and snapshot date, describe the customer using only the purchases BEFORE the cutoff date.
**Input:** `sales.parquet`, `returns.parquet` and `labels.parquet`. **Output:** `features.parquet`.

## Load the data and build an invoice-level table

In [1]:
import pandas as pd

sales = pd.read_parquet("../data/processed/sales.parquet")
returns = pd.read_parquet("../data/processed/returns.parquet")
labels = pd.read_parquet("../data/processed/labels.parquet")

# Sort by date once, so "last value" lookups later (country of the last purchase) are correct
sales = sales.sort_values("invoicedate").reset_index(drop=True)

# Revenue of each sales row = quantity x price. Value of each return row (quantity is negative, so take the absolute value)
sales["line_total"] = sales["quantity"] * sales["price"]
returns["return_value"] = (returns["quantity"] * returns["price"]).abs()

# One row per invoice (one purchase = one invoice)
invoices = (
    sales.groupby(["customer_id", "invoice"])
    .agg(
        invoice_date=("invoicedate", "min"),
        revenue=("line_total", "sum"),
        lines=("stockcode", "size"),    # number of product lines in the invoice
        items=("quantity", "sum"),      # number of items in the invoice
    )
    .reset_index()
)

# The calendar day of each invoice (several invoices on one day count as one purchase day)
invoices["invoice_day"] = invoices["invoice_date"].dt.normalize()

print("Sales:", sales.shape, "| Returns:", returns.shape, "| Labels:", labels.shape)
print("Invoices:", f"{len(invoices):,}", "| Customers:", f"{invoices['customer_id'].nunique():,}")

Sales: (775278, 10) | Returns: (16285, 10) | Labels: (36297, 4)
Invoices: 36,480 | Customers: 5,851


## Build the features of every snapshot

In [2]:
# The 19 features, in the order we want them
feature_columns = [
    "recency_days", "n_invoices", "n_purchase_days", "n_invoices_90d", "n_invoices_180d", "invoices_trend",
    "monetary_total", "median_invoice_value", "monetary_90d", "monetary_180d", "tenure_days",
    "avg_gap_days", "overdue_ratio", "avg_items_per_invoice", "avg_lines_per_invoice", "n_products",
    "n_return_lines", "return_rate", "is_uk",
]
parts = []

for cutoff in sorted(labels["cutoff"].unique()):
    cutoff = pd.Timestamp(cutoff)
    ids = labels.loc[labels["cutoff"] == cutoff, "customer_id"]

    # PAST ONLY: every table is cut at the cutoff date, then limited to the customers of this snapshot
    inv = invoices[(invoices["invoice_date"] < cutoff) & invoices["customer_id"].isin(ids)]
    sal = sales[(sales["invoicedate"] < cutoff) & sales["customer_id"].isin(ids)]
    ret = returns[(returns["invoicedate"] < cutoff) & returns["customer_id"].isin(ids)]

    # One row per customer: basic numbers from the invoices
    f = inv.groupby("customer_id").agg(
        last_purchase=("invoice_date", "max"),
        first_purchase=("invoice_date", "min"),
        first_day=("invoice_day", "min"),
        last_day=("invoice_day", "max"),
        n_invoices=("invoice", "size"),
        n_purchase_days=("invoice_day", "nunique"),
        monetary_total=("revenue", "sum"),
        median_invoice_value=("revenue", "median"),
        avg_items_per_invoice=("items", "mean"),
        avg_lines_per_invoice=("lines", "mean"),
    )

    # Recent activity: last 90 and last 180 days before the cutoff
    for days in [90, 180]:
        recent = inv[inv["invoice_date"] >= cutoff - pd.Timedelta(days=days)].groupby("customer_id")
        f[f"n_invoices_{days}d"] = recent.size()
        f[f"monetary_{days}d"] = recent["revenue"].sum()

    # Number of different products, and the country of the last purchase
    f["n_products"] = sal.groupby("customer_id")["stockcode"].nunique()
    f["is_uk"] = (sal.groupby("customer_id")["country"].last() == "United Kingdom").astype(int)

    # Returns before the cutoff
    f["n_return_lines"] = ret.groupby("customer_id").size()
    f["return_value"] = ret.groupby("customer_id")["return_value"].sum()

    # Customers without recent purchases or returns get 0
    empty_to_zero = ["n_invoices_90d", "monetary_90d", "n_invoices_180d", "monetary_180d", "n_return_lines", "return_value"]
    f[empty_to_zero] = f[empty_to_zero].fillna(0)

    # Features that are calculated from the numbers above
    f["recency_days"] = (cutoff - f["last_purchase"]).dt.days
    f["tenure_days"] = (cutoff - f["first_purchase"]).dt.days
    f["invoices_trend"] = f["n_invoices_90d"] - (f["n_invoices_180d"] - f["n_invoices_90d"])    # last 90 days minus the 90 days before
    f["avg_gap_days"] = (f["last_day"] - f["first_day"]).dt.days / (f["n_purchase_days"] - 1)   # empty for customers with one purchase day
    f["overdue_ratio"] = f["recency_days"] / f["avg_gap_days"]
    f["return_rate"] = f["return_value"] / f["monetary_total"]

    f = f.reset_index()
    f["cutoff"] = cutoff
    parts.append(f[["customer_id", "cutoff"] + feature_columns])

features = pd.concat(parts, ignore_index=True)

# Count columns are whole numbers (they became decimals because empty values were filled with 0)
count_cols = ["n_invoices_90d", "n_invoices_180d", "invoices_trend", "n_return_lines"]
features[count_cols] = features[count_cols].astype(int)

# Put the label and the split next to the features
data = features.merge(
    labels[["customer_id", "cutoff", "split", "churn"]],
    on=["customer_id", "cutoff"], how="left", validate="one_to_one",
)

print("Rows:", f"{len(data):,}", "| Rows in labels:", f"{len(labels):,}")
print("Rows without a label:", data["churn"].isna().sum())
print("Empty values (only the two gap features may have them):")
print(data.isna().sum()[data.isna().sum() > 0])

Rows: 36,297 | Rows in labels: 36,297
Rows without a label: 0
Empty values (only the two gap features may have them):
avg_gap_days     10205
overdue_ratio    10205
dtype: int64


## Save the features

In [3]:
data.to_parquet("../data/processed/features.parquet", index=False)
print("Saved features:", data.shape)

Saved features: (36297, 23)
